# 02  Enriquecimiento externo

**Proyecto:** Siniestros fatales de tr?nsito en el Per? 2021-2025  
**Curso:** Agentes Inteligentes | Grupo 5

Este notebook toma el dataset limpio generado en el notebook 01 (`siniestros_limpio.csv`) y lo enriquece con fuentes externas reales:

1. **MTC Red Vial Nacional 2022-2025**, cruzada por `COD CARRETERA`.
2. **IDH PNUD/IPE 2019**, cruzado por `DEPARTAMENTO + PROVINCIA + DISTRITO` cuando es posible.
3. **MTC Infraestructura vial por departamento 2025**, cruzada por `DEPARTAMENTO`.

El resultado se exporta a `data/procesada/siniestros_enriquecido.csv` junto con una tabla de trazabilidad.

## 0. Configuración

El notebook est? pensado para ejecutarse desde la ra?z del repositorio. No usa rutas absolutas ni tablas embebidas: todas las fuentes externas se leen desde `data/externas/`.

In [ ]:
from pathlib import Path
import re
import unicodedata
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

ROOT = Path.cwd()
DATA_EXT = ROOT / "data" / "externas"
DATA_PROC = ROOT / "data" / "procesada"
DATA_PROC.mkdir(parents=True, exist_ok=True)

PATHS = {
    "siniestros_raiz": ROOT / "siniestros_limpio.csv",
    "siniestros_proc": DATA_PROC / "siniestros_limpio.csv",
    "mtc_rvn": DATA_EXT / "1_Red_vial_nacional_2022-2025.csv",
    "mtc_diccionario": DATA_EXT / "1_Diccionario_Datos_Red_vial_nacional_2022-2025.xlsx",
    "idh": DATA_EXT / "IDH-y-Componentes-2003-2019.xlsx",
    "mtc_departamento": DATA_EXT / "344790-red-vial-existente-del-sistema-nacional-de-carreteras-segun-departamento-2010-jun-2025.xlsx",
    "salida_enriquecido": DATA_PROC / "siniestros_enriquecido.csv",
    "salida_trazabilidad": DATA_PROC / "trazabilidad_variables_externas.csv",
}

for nombre, ruta in PATHS.items():
    if nombre.startswith("salida"):
        continue
    print(f"{nombre:18s}: {'OK' if ruta.exists() else 'NO ENCONTRADO'} -> {ruta}")

In [ ]:
def normalizar_texto(valor):
    """Normaliza texto para cruces: may?sculas, sin tildes y sin espacios m?ltiples."""
    if pd.isna(valor):
        return ""
    texto = str(valor).strip().upper()
    texto = unicodedata.normalize("NFKD", texto).encode("ascii", "ignore").decode("ascii")
    texto = re.sub(r"\s+", " ", texto)
    return texto


def normalizar_codigo_ruta(valor):
    """Normaliza c?digos de carretera conservando guiones."""
    texto = normalizar_texto(valor)
    texto = texto.replace(" ", "")
    return texto


def moda_ponderada_por_longitud(grupo, col_valor, col_peso="LONGITUD"):
    """Devuelve la categor?a con mayor longitud acumulada dentro de una ruta."""
    tmp = grupo[[col_valor, col_peso]].dropna(subset=[col_valor]).copy()
    if tmp.empty:
        return np.nan
    tmp[col_peso] = pd.to_numeric(tmp[col_peso], errors="coerce").fillna(0)
    pesos = tmp.groupby(col_valor)[col_peso].sum().sort_values(ascending=False)
    if pesos.empty or pesos.iloc[0] == 0:
        modo = tmp[col_valor].mode()
        return modo.iloc[0] if not modo.empty else np.nan
    return pesos.index[0]


def validar_merge(df_antes, df_despues, nombre_fuente, cols_nuevas):
    """Valida integridad del merge y reporta cobertura."""
    print("=" * 72)
    print(f"VALIDACI?N POST-MERGE: {nombre_fuente}")
    print("=" * 72)
    print(f"Filas antes:   {len(df_antes):,}")
    print(f"Filas despu?s: {len(df_despues):,}")
    assert len(df_antes) == len(df_despues), "El merge cambi? la cantidad de filas. Revisar many-to-many."

    dist_antes = df_antes["CATEGORIA_SEVERIDAD"].value_counts().sort_index().to_dict()
    dist_despues = df_despues["CATEGORIA_SEVERIDAD"].value_counts().sort_index().to_dict()
    assert dist_antes == dist_despues, "Cambi? la distribuci?n de CATEGORIA_SEVERIDAD."
    print("Variable objetivo: inalterada")

    disponibles = [c for c in cols_nuevas if c in df_despues.columns]
    cobertura = df_despues[disponibles].notna().any(axis=1).mean() * 100 if disponibles else 0
    cruzados = int(df_despues[disponibles].notna().any(axis=1).sum()) if disponibles else 0
    no_cruzados = len(df_despues) - cruzados
    print(f"Cobertura fuente: {cobertura:.2f}% ({cruzados:,} cruzados; {no_cruzados:,} no cruzados)")
    print("Columnas nuevas:", disponibles)
    return cobertura, cruzados, no_cruzados


def agregar_trazabilidad(registros, variables, fuente, archivo_origen, url, llave_cruce, anio_dato, nivel_geografico, cobertura, cruzados, no_cruzados, observaciones):
    for variable in variables:
        registros.append({
            "variable": variable,
            "fuente": fuente,
            "archivo_origen": archivo_origen,
            "url_fuente": url,
            "llave_cruce": llave_cruce,
            "anio_dato": anio_dato,
            "nivel_geografico": nivel_geografico,
            "cobertura_pct": round(float(cobertura), 2),
            "registros_cruzados": int(cruzados),
            "registros_no_cruzados": int(no_cruzados),
            "observaciones": observaciones,
        })

## 1. Carga del dataset limpio

Se carga `siniestros_limpio.csv` desde la ra?z del repositorio. Si no existe, se intenta leer desde `data/procesada/`. Esta segunda opci?n se deja para futuras reorganizaciones del repositorio.

In [ ]:
if PATHS["siniestros_raiz"].exists():
    path_siniestros = PATHS["siniestros_raiz"]
elif PATHS["siniestros_proc"].exists():
    path_siniestros = PATHS["siniestros_proc"]
else:
    raise FileNotFoundError("No se encontr? siniestros_limpio.csv en la ra?z ni en data/procesada/.")

try:
    df = pd.read_csv(path_siniestros, encoding="utf-8-sig")
except UnicodeDecodeError:
    df = pd.read_csv(path_siniestros, encoding="latin1")

df_original = df.copy()
filas_originales = len(df)
dist_objetivo_original = df["CATEGORIA_SEVERIDAD"].value_counts().sort_index().to_dict()

print(f"Archivo base: {path_siniestros}")
print(f"Dataset limpio: {df.shape[0]:,} filas x {df.shape[1]} columnas")
print(df[["DEPARTAMENTO", "PROVINCIA", "DISTRITO", "COD CARRETERA", "CATEGORIA_SEVERIDAD"]].head())

In [ ]:
# Llaves normalizadas auxiliares. Se eliminar?n antes de exportar.
df["_DEP_NORM"] = df["DEPARTAMENTO"].apply(normalizar_texto)
df["_PROV_NORM"] = df["PROVINCIA"].apply(normalizar_texto)
df["_DIST_NORM"] = df["DISTRITO"].apply(normalizar_texto)
df["_COD_CARRETERA_NORM"] = df["COD CARRETERA"].apply(normalizar_codigo_ruta)

print(df[["DEPARTAMENTO", "PROVINCIA", "DISTRITO", "COD CARRETERA", "_COD_CARRETERA_NORM"]].head())

## 2. Fuente MTC: Red Vial Nacional 2022-2025

**Relaci?n con el problema:** las caracter?sticas de la infraestructura vial pueden influir en la severidad del siniestro. Esta fuente permite agregar informaci?n oficial del SINAC, como superficie, estado, n?mero de carriles, concesi?n, corredor log?stico y longitud de ruta.

**Llave de cruce:** `COD CARRETERA` del ONSV contra `CODIGO_RUTA` del MTC. La cobertura no ser? 100% porque el dataset del ONSV incluye v?as urbanas, provinciales, rutas sin clasificar y registros con `NO CORRESPONDE`.

In [ ]:
mtc = pd.read_csv(PATHS["mtc_rvn"], encoding="latin1", sep=None, engine="python")
print(f"MTC Red Vial Nacional: {mtc.shape[0]:,} filas x {mtc.shape[1]} columnas")
print(mtc.columns.tolist())
mtc.head(3)

In [ ]:
mtc = mtc.copy()
mtc["_COD_RUTA_NORM"] = mtc["CODIGO_RUTA"].apply(normalizar_codigo_ruta)
mtc["LONGITUD"] = pd.to_numeric(mtc["LONGITUD"], errors="coerce")
mtc["NRO_CARRILES"] = pd.to_numeric(mtc["NRO_CARRILES"], errors="coerce")
mtc["ES_CONCES"] = pd.to_numeric(mtc["ES_CONCES"], errors="coerce")

mtc_agg = (
    mtc.groupby("_COD_RUTA_NORM", dropna=False)
    .apply(lambda g: pd.Series({
        "MTC_RVN_LONGITUD_KM": g["LONGITUD"].sum(min_count=1),
        "MTC_RVN_N_CARRILES_MEDIANA": g["NRO_CARRILES"].median(),
        "MTC_RVN_SUPERFICIE_DOMINANTE": moda_ponderada_por_longitud(g, "SUPERFICIE_L"),
        "MTC_RVN_ESTADO_DOMINANTE": moda_ponderada_por_longitud(g, "ESTADO_L"),
        "MTC_RVN_ES_CONCESIONADA": int((g["ES_CONCES"].fillna(0) > 0).any()),
        "MTC_RVN_CONCESION_PRINCIPAL": moda_ponderada_por_longitud(g, "NOMBRE_CONCESION"),
        "MTC_RVN_CORREDOR_LOGISTICO": moda_ponderada_por_longitud(g, "CORREDOR_LOGISTICO"),
        "MTC_RVN_N_TRAMOS": len(g),
    }))
    .reset_index()
)

# Evitar strings vac?os como dato v?lido en concesi?n/corredor.
for col in ["MTC_RVN_CONCESION_PRINCIPAL", "MTC_RVN_CORREDOR_LOGISTICO"]:
    mtc_agg[col] = mtc_agg[col].replace({"": np.nan, " ": np.nan})

print(f"Rutas MTC agregadas: {mtc_agg.shape[0]:,}")
mtc_agg.head()

In [ ]:
cols_mtc_rvn = [c for c in mtc_agg.columns if c.startswith("MTC_RVN_")]
df_antes = df.copy()
df = df.merge(mtc_agg, left_on="_COD_CARRETERA_NORM", right_on="_COD_RUTA_NORM", how="left")
df = df.drop(columns=["_COD_RUTA_NORM"], errors="ignore")

cov_mtc_rvn, cruz_mtc_rvn, no_mtc_rvn = validar_merge(df_antes, df, "MTC Red Vial Nacional 2022-2025", cols_mtc_rvn)

print("\nEjemplos sin cruce MTC RVN:")
df.loc[df[cols_mtc_rvn].notna().any(axis=1) == False, ["COD CARRETERA", "RED VIAL", "DEPARTAMENTO"]].drop_duplicates().head(10)

## 3. Fuente IDH PNUD/IPE 2019

**Relaci?n con el problema:** el IDH y sus componentes funcionan como variables de contexto territorial: desarrollo humano, educaci?n, esperanza de vida e ingreso pueden aproximar condiciones de acceso a servicios, respuesta institucional y vulnerabilidad social.

**Llave de cruce:** se prioriza `DEPARTAMENTO + PROVINCIA + DISTRITO`, usando el archivo oficial del IPE/PNUD. El IDH 2019 es previo al periodo de siniestros 2021-2025, por lo que se interpreta como indicador estructural del territorio.

In [ ]:
idh_raw = pd.read_excel(PATHS["idh"], sheet_name="IDH 2019", header=None)
print(f"IDH raw: {idh_raw.shape[0]:,} filas x {idh_raw.shape[1]} columnas")
idh_raw.head(15)

In [ ]:
# El archivo IDH 2019 tiene jerarqu?a: departamento, provincia y distrito en filas sucesivas.
# A partir de la fila 8 est?n los datos. Las columnas 0,1,2 contienen UBIGEO / nivel / nombre.
idh = idh_raw.iloc[8:, [0, 1, 2, 5, 6, 7, 8, 9, 16]].copy()
idh.columns = [
    "UBIGEO", "NIVEL_O_DEP", "NOMBRE", "IDH_POBLACION_2019", "IDH_ESP_VIDA_2019",
    "IDH_EDUC_SECUND_PCT_2019", "IDH_ANIOS_EDUC_2019", "IDH_INGRESO_PERCAP_2019", "IDH_2019"
]
idh = idh.dropna(subset=["UBIGEO"]).copy()
idh["UBIGEO"] = pd.to_numeric(idh["UBIGEO"], errors="coerce").astype("Int64").astype(str).str.zfill(6)

for col in ["IDH_POBLACION_2019", "IDH_ESP_VIDA_2019", "IDH_EDUC_SECUND_PCT_2019", "IDH_ANIOS_EDUC_2019", "IDH_INGRESO_PERCAP_2019", "IDH_2019"]:
    idh[col] = pd.to_numeric(idh[col], errors="coerce")

idh["TIPO_NIVEL"] = np.select(
    [idh["UBIGEO"].str.endswith("0000"), idh["UBIGEO"].str.endswith("00")],
    ["DEPARTAMENTO", "PROVINCIA"],
    default="DISTRITO"
)

idh["DEPARTAMENTO_IDH"] = np.where(idh["TIPO_NIVEL"].eq("DEPARTAMENTO"), idh["NIVEL_O_DEP"], np.nan)
idh["DEPARTAMENTO_IDH"] = idh["DEPARTAMENTO_IDH"].ffill()
idh["PROVINCIA_IDH"] = np.where(idh["TIPO_NIVEL"].eq("PROVINCIA"), idh["NOMBRE"], np.nan)
idh["PROVINCIA_IDH"] = idh["PROVINCIA_IDH"].ffill()
idh["DISTRITO_IDH"] = np.where(idh["TIPO_NIVEL"].eq("DISTRITO"), idh["NOMBRE"], np.nan)

idh_dist = idh[idh["TIPO_NIVEL"].eq("DISTRITO")].copy()
idh_dist["_DEP_NORM"] = idh_dist["DEPARTAMENTO_IDH"].apply(normalizar_texto)
idh_dist["_PROV_NORM"] = idh_dist["PROVINCIA_IDH"].apply(normalizar_texto)
idh_dist["_DIST_NORM"] = idh_dist["DISTRITO_IDH"].apply(normalizar_texto)

cols_idh = [
    "IDH_2019", "IDH_ESP_VIDA_2019", "IDH_EDUC_SECUND_PCT_2019",
    "IDH_ANIOS_EDUC_2019", "IDH_INGRESO_PERCAP_2019", "IDH_POBLACION_2019"
]
idh_dist_sel = idh_dist[["_DEP_NORM", "_PROV_NORM", "_DIST_NORM", "UBIGEO"] + cols_idh].drop_duplicates(["_DEP_NORM", "_PROV_NORM", "_DIST_NORM"])
idh_dist_sel = idh_dist_sel.rename(columns={"UBIGEO": "IDH_UBIGEO"})

print(f"Distritos IDH 2019 disponibles: {idh_dist_sel.shape[0]:,}")
idh_dist_sel.head()

In [ ]:
df_antes = df.copy()
df = df.merge(idh_dist_sel, on=["_DEP_NORM", "_PROV_NORM", "_DIST_NORM"], how="left")

cov_idh, cruz_idh, no_idh = validar_merge(df_antes, df, "IDH PNUD/IPE 2019 distrital", cols_idh)

print("\nEjemplos sin cruce IDH:")
df.loc[df["IDH_2019"].isna(), ["DEPARTAMENTO", "PROVINCIA", "DISTRITO"]].drop_duplicates().head(15)

## 4. Fuente MTC: infraestructura vial por departamento 2025

**Relaci?n con el problema:** esta fuente agrega el contexto vial departamental: longitud total de red, red nacional/departamental/vecinal y proporci?n pavimentada. Es una variable de exposici?n e infraestructura a nivel territorial.

**Llave de cruce:** `DEPARTAMENTO`. Se usa la hoja 2025 porque es el corte m?s reciente disponible en el archivo descargado del MTC.

In [ ]:
mtc_dep_raw = pd.read_excel(PATHS["mtc_departamento"], sheet_name="2025", header=None)
print(f"MTC infraestructura departamental raw: {mtc_dep_raw.shape[0]:,} filas x {mtc_dep_raw.shape[1]} columnas")
mtc_dep_raw.head(20)

In [ ]:
# En este Excel, la tabla empieza en la columna B y tiene columnas vac?as separadoras.
# Se seleccionan: departamento, total, nacional total/pav/no pav, departamental total/pav/no pav, vecinal total/pav/no pav.
cols_excel = [1, 2, 4, 5, 6, 8, 9, 10, 12, 13, 14]
mtc_dep = mtc_dep_raw.iloc[9:, cols_excel].copy()
mtc_dep.columns = [
    "DEPARTAMENTO_MTC_DEP", "MTC_DEP_RED_TOTAL_KM", "MTC_DEP_NACIONAL_TOTAL_KM",
    "MTC_DEP_NACIONAL_PAV_KM", "MTC_DEP_NACIONAL_NO_PAV_KM",
    "MTC_DEP_DEPARTAMENTAL_TOTAL_KM", "MTC_DEP_DEPARTAMENTAL_PAV_KM", "MTC_DEP_DEPARTAMENTAL_NO_PAV_KM",
    "MTC_DEP_VECINAL_TOTAL_KM", "MTC_DEP_VECINAL_PAV_KM", "MTC_DEP_VECINAL_NO_PAV_KM",
]
mtc_dep = mtc_dep.dropna(subset=["DEPARTAMENTO_MTC_DEP"]).copy()
mtc_dep = mtc_dep[~mtc_dep["DEPARTAMENTO_MTC_DEP"].astype(str).str.upper().str.contains("FUENTE|ELABORACION|NOTA", na=False)].copy()
mtc_dep["_DEP_NORM"] = mtc_dep["DEPARTAMENTO_MTC_DEP"].apply(normalizar_texto)

cols_mtc_dep = [c for c in mtc_dep.columns if c.startswith("MTC_DEP_")]
for col in cols_mtc_dep:
    mtc_dep[col] = pd.to_numeric(mtc_dep[col], errors="coerce")

mtc_dep["MTC_DEP_PAV_TOTAL_KM"] = mtc_dep["MTC_DEP_NACIONAL_PAV_KM"] + mtc_dep["MTC_DEP_DEPARTAMENTAL_PAV_KM"] + mtc_dep["MTC_DEP_VECINAL_PAV_KM"]
mtc_dep["MTC_DEP_NO_PAV_TOTAL_KM"] = mtc_dep["MTC_DEP_NACIONAL_NO_PAV_KM"] + mtc_dep["MTC_DEP_DEPARTAMENTAL_NO_PAV_KM"] + mtc_dep["MTC_DEP_VECINAL_NO_PAV_KM"]
mtc_dep["MTC_DEP_PCT_PAVIMENTADA"] = np.where(
    mtc_dep["MTC_DEP_RED_TOTAL_KM"] > 0,
    mtc_dep["MTC_DEP_PAV_TOTAL_KM"] / mtc_dep["MTC_DEP_RED_TOTAL_KM"] * 100,
    np.nan,
)
cols_mtc_dep_final = [
    "MTC_DEP_RED_TOTAL_KM", "MTC_DEP_NACIONAL_TOTAL_KM", "MTC_DEP_DEPARTAMENTAL_TOTAL_KM",
    "MTC_DEP_VECINAL_TOTAL_KM", "MTC_DEP_PAV_TOTAL_KM", "MTC_DEP_NO_PAV_TOTAL_KM",
    "MTC_DEP_PCT_PAVIMENTADA"
]
mtc_dep_sel = mtc_dep[["_DEP_NORM"] + cols_mtc_dep_final].drop_duplicates("_DEP_NORM")

print(f"Departamentos MTC 2025: {mtc_dep_sel.shape[0]}")
mtc_dep_sel.head()

In [ ]:
df_antes = df.copy()
df = df.merge(mtc_dep_sel, on="_DEP_NORM", how="left")

cov_mtc_dep, cruz_mtc_dep, no_mtc_dep = validar_merge(df_antes, df, "MTC infraestructura vial por departamento 2025", cols_mtc_dep_final)

print("\nEjemplos sin cruce MTC departamento:")
df.loc[df["MTC_DEP_RED_TOTAL_KM"].isna(), ["DEPARTAMENTO"]].drop_duplicates().head(10)

## 5. Fuentes evaluadas y no integradas en esta versi?n

- **INEI / EstaDist:** se evalu? para indicadores distritales, pero el portal requiere selecci?n manual de distritos para exportaci?n. Se deja como mejora futura para no introducir un proceso manual incompleto.
- **SENAMHI:** se evalu? para clima objetivo, pero requiere descarga por estaci?n y cruce espaciotemporal por coordenadas y fecha exacta. El dataset limpio conserva a?o/mes/hora, pero no la fecha completa original.
- **MTC flujo vehicular por peajes:** ?til como proxy de exposici?n vehicular, pero su cruce correcto requiere peajes georreferenciados o agregaci?n por ruta/departamento. Se deja como mejora futura.

## 6. Trazabilidad de variables externas

Se documenta cada variable nueva con fuente, archivo origen, URL, llave de cruce, nivel geogr?fico, cobertura y observaciones.

In [ ]:
trazabilidad = []

agregar_trazabilidad(
    trazabilidad, cols_mtc_rvn,
    fuente="MTC - Red Vial Nacional SINAC 2022-2025",
    archivo_origen=PATHS["mtc_rvn"].as_posix(),
    url="https://www.datosabiertos.gob.pe/dataset/red-vial-nacional-del-sistema-nacional-de-carreteras-2022-2025-ministerio-de-transportes-y-comunicaciones-mtc",
    llave_cruce="COD CARRETERA -> CODIGO_RUTA",
    anio_dato="2022-2025",
    nivel_geografico="Ruta / tramo agregado por c?digo de ruta",
    cobertura=cov_mtc_rvn,
    cruzados=cruz_mtc_rvn,
    no_cruzados=no_mtc_rvn,
    observaciones="Cobertura parcial esperada: no aplica a v?as urbanas, NO CORRESPONDE, SIN CLASIFICAR o rutas no nacionales."
)

agregar_trazabilidad(
    trazabilidad, cols_idh + ["IDH_UBIGEO"],
    fuente="PNUD/IPE - IDH y componentes 2019",
    archivo_origen=PATHS["idh"].as_posix(),
    url="https://ipe.org.pe/indice-de-desarrollo-humano-idh/",
    llave_cruce="DEPARTAMENTO + PROVINCIA + DISTRITO",
    anio_dato="2019",
    nivel_geografico="Distrito",
    cobertura=cov_idh,
    cruzados=cruz_idh,
    no_cruzados=no_idh,
    observaciones="Indicador estructural previo al periodo 2021-2025; puede haber no cruces por diferencias de nomenclatura distrital."
)

agregar_trazabilidad(
    trazabilidad, cols_mtc_dep_final,
    fuente="MTC - Infraestructura vial existente del SINAC por departamento",
    archivo_origen=PATHS["mtc_departamento"].as_posix(),
    url="https://www.gob.pe/institucion/mtc/informes-publicaciones/344790-estadistica-infraestructura-de-transportes-infraestructura-vial",
    llave_cruce="DEPARTAMENTO",
    anio_dato="2025 (junio)",
    nivel_geografico="Departamento",
    cobertura=cov_mtc_dep,
    cruzados=cruz_mtc_dep,
    no_cruzados=no_mtc_dep,
    observaciones="Variables agregadas a nivel departamental; no capturan heterogeneidad intradepartamental."
)

df_trazabilidad = pd.DataFrame(trazabilidad)
df_trazabilidad

## 7. Validaci?n global y exportaci?n

Antes de exportar se verifica que no se hayan duplicado filas ni alterado la variable objetivo. Las columnas auxiliares de normalizaci?n se eliminan del CSV final.

In [ ]:
cols_aux = [c for c in df.columns if c.startswith("_")]
df_final = df.drop(columns=cols_aux, errors="ignore").copy()

assert len(df_final) == filas_originales, "La cantidad de filas cambi? durante el enriquecimiento."
assert df_final["CATEGORIA_SEVERIDAD"].value_counts().sort_index().to_dict() == dist_objetivo_original, "La variable objetivo cambi?."

cols_originales = list(df_original.columns)
cols_nuevas = [c for c in df_final.columns if c not in cols_originales]

print("=" * 72)
print("VALIDACI?N GLOBAL FINAL")
print("=" * 72)
print(f"Filas finales: {len(df_final):,}")
print(f"Columnas originales: {len(cols_originales)}")
print(f"Columnas nuevas: {len(cols_nuevas)}")
print(f"Columnas finales: {df_final.shape[1]}")
print("\nVariables nuevas:")
for col in cols_nuevas:
    print(f"- {col}: {df_final[col].notna().mean() * 100:.2f}% cobertura")

# Exportar con utf-8-sig para que Excel reconozca tildes y ?.
df_final.to_csv(PATHS["salida_enriquecido"], index=False, encoding="utf-8-sig")
df_trazabilidad.to_csv(PATHS["salida_trazabilidad"], index=False, encoding="utf-8-sig")

print("\nArchivos generados:")
print(PATHS["salida_enriquecido"])
print(PATHS["salida_trazabilidad"])
print("\nPr?ximo paso: usar data/procesada/siniestros_enriquecido.csv como entrada del notebook 03.")

## 8. Resumen metodol?gico

El dataset enriquecido conserva todos los registros originales y agrega variables externas reales. La cobertura de MTC Red Vial Nacional es parcial por la propia naturaleza de la fuente: solo representa rutas nacionales del SINAC y no cubre v?as urbanas o registros sin c?digo de carretera. El IDH se integra a nivel distrital cuando los nombres coinciden tras normalizaci?n, mientras que la infraestructura vial departamental entrega contexto territorial agregado con cobertura esperada cercana al 100%.